# Time Series Analysis and Forecasting of Electricity Demand in Kenya

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA

import warnings
warnings.filterwarnings("ignore")

In [7]:
# Load the datasets
demand_df = pd.read_csv("../data/kenya-electricity-demand.csv")

generation_df = pd.read_csv("../data/kenya-electricity-generation.csv")

percapita_df = pd.read_csv("../data/kenya-per-capita-electricity-generation.csv")

population_df = pd.read_csv("../data/kenya-population.csv")

In [8]:
# Preview the datasets
print("Electricity Demand Dataset")
display(demand_df.head())

print("Electricity Generation Dataset")
display(generation_df.head())

print("Per Capita Generation Dataset")
display(percapita_df.head())

print("Population Dataset")
display(population_df.head())

Electricity Demand Dataset


,Country,Code,Year,Electricity-demand
0,Kenya,KEN,2000,4.51
1,Kenya,KEN,2001,4.98
2,Kenya,KEN,2002,5.37
3,Kenya,KEN,2003,5.67
4,Kenya,KEN,2004,6.32


Electricity Generation Dataset


,Country,Code,Year,Coal,Fossil-fuels,Gas,Hydro,Low-carbon,Nuclear,Oil,Renewables,Solar,Solar-wind,Wind,Geothermal,Total-electricity-produced
0,Kenya,KEN,2000,0.0,2.57,0.0,1.31,1.74,0.0,2.57,1.74,0.0,0.0,0.0,0.43,4.31
1,Kenya,KEN,2001,0.0,1.95,0.0,2.38,2.86,0.0,1.95,2.86,0.0,0.0,0.0,0.48,4.81
2,Kenya,KEN,2002,0.0,1.20,0.0,3.09,3.95,0.0,1.20,3.95,0.0,0.0,0.0,0.86,5.15
3,Kenya,KEN,2003,0.0,0.99,0.0,3.23,4.49,0.0,0.99,4.49,0.0,0.0,0.0,1.26,5.48
4,Kenya,KEN,2004,0.0,1.82,0.0,2.84,4.34,0.0,1.82,4.34,0.0,0.0,0.0,1.50,6.16


Per Capita Generation Dataset


,Country,Code,Year,Coal-per-capita,Fossil-fuels-per-capita,Gas-per-capita,Hydro-per-capita,Low-carbon-per-capita,Nuclear-per-capita,Oil-per-capita,Renewable-per-capita,Solar-per-capita,Solar-wind-per-capita,Wind-per-capita,Geothermal-per-capita,Total-electricity-per-capita-produced
0,Kenya,KEN,2000,0.0,83.869360,0.0,42.750530,56.783150,0.0,83.869360,56.783150,0.0,0.0,0.0,14.03262,140.652510
1,Kenya,KEN,2001,0.0,61.671448,0.0,75.270800,90.451454,0.0,61.671448,90.451454,0.0,0.0,0.0,15.18065,152.122902
2,Kenya,KEN,2002,0.0,36.776188,0.0,94.698685,121.054955,0.0,36.776188,121.054955,0.0,0.0,0.0,26.35627,157.831143
3,Kenya,KEN,2003,0.0,29.418552,0.0,95.981740,133.423540,0.0,29.418552,133.423540,0.0,0.0,0.0,37.44180,162.842092
4,Kenya,KEN,2004,0.0,52.429240,0.0,81.812660,125.023580,0.0,52.429240,125.023580,0.0,0.0,0.0,43.21092,177.452820


Population Dataset


,Country,Year,Population
0,Kenya,2000,30642894
1,Kenya,2001,31619171
2,Kenya,2002,32629809
3,Kenya,2003,33652233
4,Kenya,2004,34713452


In [9]:
# Check dataset shapes
print("Demand Shape:", demand_df.shape)
print("Generation Shape:", generation_df.shape)
print("Per Capita Shape:", percapita_df.shape)
print("Population Shape:", population_df.shape)

Demand Shape: (25, 4)
Generation Shape: (25, 16)
Per Capita Shape: (25, 16)
Population Shape: (25, 3)


In [10]:
# Checking dataset info
print("Demand Dataset Info")
demand_df.info()


Demand Dataset Info
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25 entries, 0 to 24
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Country             25 non-null     object 
 1   Code                25 non-null     object 
 2   Year                25 non-null     int64  
 3   Electricity-demand  25 non-null     float64
dtypes: float64(1), int64(1), object(2)
memory usage: 932.0+ bytes


In [11]:
generation_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25 entries, 0 to 24
Data columns (total 16 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Country                     25 non-null     object 
 1   Code                        25 non-null     object 
 2   Year                        25 non-null     int64  
 3   Coal                        25 non-null     float64
 4   Fossil-fuels                25 non-null     float64
 5   Gas                         25 non-null     float64
 6   Hydro                       25 non-null     float64
 7   Low-carbon                  25 non-null     float64
 8   Nuclear                     25 non-null     float64
 9   Oil                         25 non-null     float64
 10  Renewables                  25 non-null     float64
 11  Solar                       25 non-null     float64
 12  Solar-wind                  25 non-null     float64
 13  Wind                        25 non-nu

In [12]:
percapita_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25 entries, 0 to 24
Data columns (total 16 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   Country                                25 non-null     object 
 1   Code                                   25 non-null     object 
 2   Year                                   25 non-null     int64  
 3   Coal-per-capita                        25 non-null     float64
 4   Fossil-fuels-per-capita                25 non-null     float64
 5   Gas-per-capita                         25 non-null     float64
 6   Hydro-per-capita                       25 non-null     float64
 7   Low-carbon-per-capita                  25 non-null     float64
 8   Nuclear-per-capita                     25 non-null     float64
 9   Oil-per-capita                         25 non-null     float64
 10  Renewable-per-capita                   25 non-null     float64
 11  Solar-pe

In [13]:
population_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25 entries, 0 to 24
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Country     25 non-null     object
 1   Year        25 non-null     int64 
 2   Population  25 non-null     int64 
dtypes: int64(2), object(1)
memory usage: 732.0+ bytes
